# CLAM-TR LoRA Figure Generator (F1 + F2 + F3)

Generates the three LoRA-related figures of the thesis appendix (Appendix Figures 1-3):

- **F1** -- Stage 4 12-configuration QWK bar chart (sorted by mean QWK)
- **F2** -- Frozen UNI vs LoRA-UNI paired forest plot (6 aggregators, bestHP)
- **F3** -- Stage 2 LoRA encoder direct classifier 5-fold QWK distribution

## How to run on Colab

1. Open this notebook in Google Colab (File -> Open notebook -> Upload, or use the GitHub tab if pushed).
2. Run the cells in order. The first cell mounts Google Drive.
3. Verify the JSON inputs are at `/content/drive/MyDrive/PANDA_LoRA/results/`.
4. Output PDFs + PNGs land in `/content/drive/MyDrive/PANDA_LoRA/figures/`.
5. After all cells run, copy the three PDFs to the figure folder of the thesis build:
   - `fig_lora_factorial_bars.pdf`
   - `fig_lora_vs_frozen_forest.pdf`
   - `fig_lora_stage2_folds.pdf`

No internet required; no GPU required; runs in well under a minute on Colab Free CPU.

## What this notebook reads

- `lora_training_results.json` -- Stage 2 5-fold summary (mean_qwk 0.9563)
- `v20_lora_features_results.json` -- Stage 4 12-config matrix (best 0.9529)

Both files were produced by `CLAM_TR_LoRA_v1_FIXED_v8_SHARDED.ipynb` and are
also in this repository at
`05-experimental/lora_v1_results/`.

## Frozen baselines (hardcoded in F2)

F2 uses the frozen UNI bestHP values from the thesis Appendix B Table B.1
(verified verbatim from `v20_final_summary.json`). If the frozen baselines
ever change, update the `frozen_uni` dict in the F2 cell.

## Setup -- Drive mount + path verification

In [ ]:
# Mount Google Drive (skip silently if already mounted)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, json

RESULTS_DIR = '/content/drive/MyDrive/PANDA_LoRA/results'
FIGS_DIR    = '/content/drive/MyDrive/PANDA_LoRA/figures'
os.makedirs(FIGS_DIR, exist_ok=True)

# Verify required inputs are present
required = ['lora_training_results.json', 'v20_lora_features_results.json']
print('=== Input verification ===')
for fname in required:
    fpath = os.path.join(RESULTS_DIR, fname)
    assert os.path.exists(fpath), f'MISSING: {fpath}'
    size = os.path.getsize(fpath)
    print(f'  OK  {fname:40s} {size:>10} bytes')

print(f'\nFigures will be saved to: {FIGS_DIR}')

In [ ]:
# Matplotlib setup -- PDF-embeddable serif, type-42 fonts (editable text in PDF viewers)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42   # embed editable text in PDF
matplotlib.rcParams['ps.fonttype']  = 42
matplotlib.rcParams['font.family']  = 'serif'
matplotlib.rcParams['font.size']    = 10
matplotlib.rcParams['axes.labelsize']  = 10
matplotlib.rcParams['axes.titlesize']  = 11
matplotlib.rcParams['xtick.labelsize'] = 9
matplotlib.rcParams['ytick.labelsize'] = 9
matplotlib.rcParams['legend.fontsize'] = 9
print('Matplotlib configured (PDF type-42 serif).')

## F1 -- Stage 4 12-configuration QWK bar chart

Horizontal bar chart of the 12 LoRA-feature aggregator configurations sorted
by 5-fold mean QWK descending. Error bars are 5-fold $\pm 1\sigma$. The red
dashed line marks the frozen UNI2-h `SpatialMultRetNet_bestHP` baseline
(0.9530); the shaded red band is $\pm 1\sigma$ noise envelope (0.0029).

**Thesis label:** `fig:lora_factorial_bars`. **Placement:** Appendix A new
"Reported Outcome" subsection.

In [ ]:
# F1 -- Stage 4 12-config QWK bar chart (sorted by mean QWK)
with open(os.path.join(RESULTS_DIR, 'v20_lora_features_results.json')) as f:
    v = json.load(f)

# Sort by mean QWK descending
sorted_configs = sorted(v.items(), key=lambda kv: kv[1]['mean_qwk'], reverse=True)
names = [k for k, _ in sorted_configs]
means = [vv['mean_qwk'] for _, vv in sorted_configs]
stds  = [vv['std_qwk']  for _, vv in sorted_configs]

# Frozen-UNI2-h baseline reference (thesis Appendix B Table B.1)
baseline_uni2h = 0.9530
baseline_uni2h_std = 0.0029

fig, ax = plt.subplots(figsize=(10, 6))
ypos = list(range(len(names)))[::-1]
ax.barh(ypos, means, xerr=stds, color='#4C72B0', edgecolor='black',
        linewidth=0.5, capsize=3, alpha=0.85)
ax.axvline(baseline_uni2h, color='red', linestyle='--', linewidth=1,
           label=f'Frozen UNI2-h SpatialMultRetNet baseline ({baseline_uni2h:.4f})')
ax.axvspan(baseline_uni2h - baseline_uni2h_std,
           baseline_uni2h + baseline_uni2h_std,
           color='red', alpha=0.10,
           label='Baseline +/-1 sigma noise envelope')

# Annotate each bar with mean QWK
for y, m, s in zip(ypos, means, stds):
    ax.text(m + s + 0.0008, y, f'{m:.4f}', va='center', fontsize=8)

ax.set_yticks(ypos)
ax.set_yticklabels(names)
ax.set_xlabel('5-fold mean QWK (+/-1 sigma)')
ax.set_xlim(0.925, 0.962)
ax.set_title('Stage 4: LoRA-UNI features through v20 aggregator pipeline\n'
             '(12 configurations, sorted by mean QWK)')
ax.legend(loc='lower right')
ax.grid(axis='x', linestyle=':', alpha=0.5)
plt.tight_layout()

pdf_path = os.path.join(FIGS_DIR, 'fig_lora_factorial_bars.pdf')
png_path = os.path.join(FIGS_DIR, 'fig_lora_factorial_bars.png')
plt.savefig(pdf_path, bbox_inches='tight')
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.show()

print(f'\nSaved:\n  {pdf_path}\n  {png_path}')
print(f'\nBest config: {names[0]} at {means[0]:.4f} +/- {stds[0]:.4f}')
print(f'Worst config: {names[-1]} at {means[-1]:.4f} +/- {stds[-1]:.4f}')
print(f'Range: {means[0]-means[-1]:.4f} QWK across 12 configurations')

## F2 -- Frozen UNI vs LoRA-UNI paired forest plot

Per-aggregator comparison of frozen UNI features (blue circles) vs LoRA-UNI
features (red squares) under the best-HP set. Six aggregators on the y-axis;
5-fold mean QWK + $\pm 1\sigma$ error bars on the x-axis. The per-aggregator
$\Delta$QWK (LoRA - Frozen) is annotated on the right.

**Thesis label:** `fig:lora_vs_frozen_forest`. **Placement:** Appendix A new
"Reported Outcome" subsection or Ch.4 sec:sub5_lora.

**Frozen baseline source:** thesis Appendix B Table B.1, verbatim from
`v20_final_summary.json`. If those values change, update the `frozen_uni`
dict below.

In [ ]:
# F2 -- Paired forest plot: frozen UNI vs LoRA UNI per aggregator (bestHP)
with open(os.path.join(RESULTS_DIR, 'v20_lora_features_results.json')) as f:
    v = json.load(f)

# Frozen UNI bestHP values (from thesis Appendix B Table B.1, verbatim)
frozen_uni = {
    'ABMIL':              (0.9456, 0.0039),
    'DualBranch':         (0.9499, 0.0070),
    'RetNet':             (0.9543, 0.0046),
    'SpatialAdd':         (0.9494, 0.0043),
    'SpatialMult':        (0.9386, 0.0162),
    'SpatialMultRetNet':  (0.9519, 0.0045),
}

aggs = ['ABMIL', 'DualBranch', 'RetNet', 'SpatialAdd', 'SpatialMult', 'SpatialMultRetNet']
lora_means = [v[f'{a}_bestHP']['mean_qwk'] for a in aggs]
lora_stds  = [v[f'{a}_bestHP']['std_qwk']  for a in aggs]
frozen_means = [frozen_uni[a][0] for a in aggs]
frozen_stds  = [frozen_uni[a][1] for a in aggs]

fig, ax = plt.subplots(figsize=(10, 5))
ypos = list(range(len(aggs)))
offset = 0.18
ax.errorbar(frozen_means, [y - offset for y in ypos], xerr=frozen_stds,
            fmt='o', color='#4C72B0', label='Frozen UNI',
            capsize=4, markersize=8, linewidth=1.5)
ax.errorbar(lora_means, [y + offset for y in ypos], xerr=lora_stds,
            fmt='s', color='#C44E52', label='LoRA UNI (Stage 4)',
            capsize=4, markersize=8, linewidth=1.5)

# Per-aggregator delta annotation (LoRA - Frozen)
for y, f_m, l_m in zip(ypos, frozen_means, lora_means):
    delta = l_m - f_m
    sign = '+' if delta >= 0 else ''
    color = '#888888'
    ax.text(0.965, y, f'{sign}{delta:.4f}', fontsize=9, va='center', color=color)

ax.set_yticks(ypos)
ax.set_yticklabels(aggs)
ax.set_xlabel('5-fold mean QWK (+/-1 sigma)')
ax.set_xlim(0.915, 0.975)
ax.set_title('Frozen UNI vs LoRA-UNI feature pipeline -- paired comparison (bestHP)')
ax.axvline(0.95, color='gray', linestyle=':', alpha=0.4)
ax.legend(loc='lower right')
ax.grid(axis='x', linestyle=':', alpha=0.5)
ax.text(0.972, len(aggs) - 0.3, 'delta QWK',
        fontsize=9, va='center', color='#555555', fontweight='bold')
plt.tight_layout()

pdf_path = os.path.join(FIGS_DIR, 'fig_lora_vs_frozen_forest.pdf')
png_path = os.path.join(FIGS_DIR, 'fig_lora_vs_frozen_forest.png')
plt.savefig(pdf_path, bbox_inches='tight')
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.show()

print(f'\nSaved:\n  {pdf_path}\n  {png_path}')
print('\nPer-aggregator delta QWK (LoRA - Frozen):')
for a, f_m, l_m in zip(aggs, frozen_means, lora_means):
    delta = l_m - f_m
    sign = '+' if delta >= 0 else ''
    print(f'  {a:22s}  {sign}{delta:+.4f}')

## F3 -- Stage 2 5-fold QWK distribution

Per-fold validation QWK for the LoRA encoder direct classifier (Stage 2,
end-to-end joint training with the SpatialMultRetNet head). 5 bars +
5-fold mean line + $\pm 1\sigma$ band.

**Thesis label:** `fig:lora_stage2_folds`. **Placement:** Ch.2 sec:lora_v1
status block or Appendix A new "Reported Outcome" subsection.

**Note on fold-0 wall-time anomaly:** fold-0 wall-time is logged as 6.18 min
vs ~472 min for folds 1-4. The QWK value (0.9531) is consistent with the
cluster, ruling out training failure. Likely cause: session-resume timer
reset under the single-fold-per-session workflow. For aggregate wall-time
reporting, treat fold-0 time as unrecorded; use folds 1-4 mean (~472 min).

In [ ]:
# F3 -- Stage 2 LoRA-direct classifier 5-fold QWK distribution
with open(os.path.join(RESULTS_DIR, 'lora_training_results.json')) as f:
    r = json.load(f)

fold_qwks = [fm['qwk'] for fm in r['fold_metrics']]
fold_ids  = [f"Fold {fm['fold']}" for fm in r['fold_metrics']]
mean_qwk  = r['mean_qwk']
std_qwk   = r['std_qwk']
best_fold = r['best_fold']

fig, ax = plt.subplots(figsize=(7, 5))
xpos = list(range(len(fold_ids)))
colors = ['#FFB400' if i == best_fold else '#55A868' for i in range(len(fold_ids))]
ax.bar(xpos, fold_qwks, color=colors, edgecolor='black', linewidth=0.5, alpha=0.85)
ax.axhline(mean_qwk, color='red', linestyle='--', linewidth=1.2,
           label=f'5-fold mean ({mean_qwk:.4f})')
ax.axhspan(mean_qwk - std_qwk, mean_qwk + std_qwk,
           color='red', alpha=0.12,
           label=f'+/-1 sigma ({std_qwk:.4f})')

# Per-fold QWK label above each bar
for x, q in zip(xpos, fold_qwks):
    ax.text(x, q + 0.0005, f'{q:.4f}', ha='center', va='bottom', fontsize=9)

# Best-fold marker
ax.text(best_fold, fold_qwks[best_fold] + 0.0020,
        '(best)', ha='center', va='bottom', fontsize=8,
        color='#B86B00', fontweight='bold')

ax.set_xticks(xpos)
ax.set_xticklabels(fold_ids)
ax.set_ylabel('Validation 5-fold QWK')
ax.set_ylim(0.945, 0.965)
ax.set_title('Stage 2: LoRA encoder direct classifier (5-fold cross-validation)')
ax.legend(loc='lower right')
ax.grid(axis='y', linestyle=':', alpha=0.5)
plt.tight_layout()

pdf_path = os.path.join(FIGS_DIR, 'fig_lora_stage2_folds.pdf')
png_path = os.path.join(FIGS_DIR, 'fig_lora_stage2_folds.png')
plt.savefig(pdf_path, bbox_inches='tight')
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.show()

print(f'\nSaved:\n  {pdf_path}\n  {png_path}')
print(f'\n5-fold mean QWK : {mean_qwk:.16f}  (in prose: {mean_qwk:.4f})')
print(f'5-fold std QWK  : {std_qwk:.16f}  (in prose: {std_qwk:.4f})')
print(f'Best fold       : {best_fold} (QWK {fold_qwks[best_fold]:.4f})')
for i, (fid, q) in enumerate(zip(fold_ids, fold_qwks)):
    mark = ' <-- best' if i == best_fold else ''
    print(f'  {fid:8s}  {q:.4f}{mark}')

## Done -- next steps

All 3 figures saved to `/content/drive/MyDrive/PANDA_LoRA/figures/`:
- `fig_lora_factorial_bars.{pdf,png}`
- `fig_lora_vs_frozen_forest.{pdf,png}`
- `fig_lora_stage2_folds.{pdf,png}`

## Copy to the thesis build (run on your local machine, not in Colab)

On Windows PowerShell:

```powershell
$src = '<YOUR_DRIVE_DIR>\PANDA_LoRA\figures'
$dst = '<YOUR_PROJECT_DIR>\thesis\main\figures'
Copy-Item "$src\fig_lora_factorial_bars.pdf"   -Destination $dst
Copy-Item "$src\fig_lora_vs_frozen_forest.pdf" -Destination $dst
Copy-Item "$src\fig_lora_stage2_folds.pdf"     -Destination $dst
```

Or on Git Bash:

```bash
cp "<YOUR_DRIVE_DIR>/PANDA_LoRA/figures/fig_lora_factorial_bars.pdf"   "<YOUR_PROJECT_DIR>/thesis/main/figures/"
cp "<YOUR_DRIVE_DIR>/PANDA_LoRA/figures/fig_lora_vs_frozen_forest.pdf" "<YOUR_PROJECT_DIR>/thesis/main/figures/"
cp "<YOUR_DRIVE_DIR>/PANDA_LoRA/figures/fig_lora_stage2_folds.pdf"     "<YOUR_PROJECT_DIR>/thesis/main/figures/"
```

The thesis includes the three figures with the LaTeX labels `fig:lora_factorial_bars`, `fig:lora_vs_frozen_forest` and `fig:lora_stage2_folds` (Appendix Figures 1-3); the printed versions are re-set by `06-thesis-figures/plots/make_appendix_figs_1_2_3.py`.